# S1 — NumPy 與向量化思維

> **時間**：2 小時（講授 70min + 課堂練習 40min + QA 10min）  
> **資料**：`datasets/ecommerce/products.csv`  
> **學完能幹嘛**：用「一行程式」取代 for-loop，處理上百萬筆數字比 Excel 快 100 倍

---

## 為什麼資料人一定要會 NumPy？

做資料分析時，你會遇到的場景：

- 要把 **10 萬筆訂單金額**同時打 9 折 → 用 for-loop 要寫 5 行、跑 3 秒；用 NumPy **1 行、0.01 秒**
- 要算**每個商品的庫存總值**（價格 × 庫存）→ NumPy 一個乘號就搞定
- Pandas、scikit-learn、TensorFlow 底層**全部**都是 NumPy

**一句話口訣：寫 for-loop 處理數字的那一刻，你就輸了。**


---
## 核心觀念 1／3 — ndarray：N 維陣列

**定義**：ndarray 是一個「同型別、固定大小」的多維陣列。跟 Python list 的差別：

| 比較項 | Python list | NumPy ndarray |
|---|---|---|
| 元素型別 | 可混合 | 必須同型別 |
| 速度 | 慢 | 快 10~100 倍 |
| 記憶體 | 耗 | 省 |
| 向量化運算 | ❌ 要 for-loop | ✅ 直接運算 |


In [ ]:
import numpy as np

# 從 list 建立
arr = np.array([1, 2, 3, 4, 5])
print('arr       =', arr)
print('shape     =', arr.shape)    # (5,)  → 一維，5 個元素
print('dtype     =', arr.dtype)    # int64
print('ndim      =', arr.ndim)     # 1

# 二維陣列（矩陣）
mat = np.array([[1, 2, 3], [4, 5, 6]])
print('\nmat       =\n', mat)
print('mat.shape =', mat.shape)    # (2, 3)

# 常用快速建立
print('\nzeros:', np.zeros(5))
print('ones :', np.ones(3))
print('range:', np.arange(0, 10, 2))  # 0~10 每隔 2
print('linsp:', np.linspace(0, 1, 5)) # 0~1 均分 5 個

**口訣**：`np.array()` 把 list 變陣列，`.shape` 看形狀，`.dtype` 看型別。


---
## 核心觀念 2／3 — 索引、切片、布林遮罩

三種取值方法，**布林遮罩**是資料分析最常用的。


In [ ]:
a = np.array([10, 20, 30, 40, 50, 60])

# 1) 單點索引（和 list 一樣）
print('a[0]   =', a[0])    # 10
print('a[-1]  =', a[-1])   # 60（倒數第一個）

# 2) 切片 [start:stop:step]
print('a[1:4] =', a[1:4])  # [20 30 40]
print('a[::2] =', a[::2])  # [10 30 50]（每隔一個）

# 3) 布林遮罩（最重要！）
mask = a > 30
print('\nmask       =', mask)   # [F F F T T T]
print('a[mask]    =', a[mask])  # [40 50 60]
print('a[a > 30]  =', a[a > 30])  # 一步到位

# 多條件：用 & (and), | (or)，記得加括號！
print('a[(a > 20) & (a < 60)] =', a[(a > 20) & (a < 60)])

**口訣**：`a[條件]` 直接篩，多條件要加**括號**、用 `&` `|`（不是 `and` `or`）。


---
## 核心觀念 3／3 — 向量化運算與廣播

**向量化 (vectorization)**：陣列之間直接做數學運算，**不用 for-loop**。  
**廣播 (broadcasting)**：形狀不同的陣列也能運算，NumPy 會自動補齊。


In [ ]:
# 向量化：兩個陣列逐元素相加 / 相乘
prices = np.array([100, 200, 300, 400])
qty    = np.array([  2,   1,   3,   5])

total = prices * qty        # → [200, 200, 900, 2000]
print('total =', total)
print('合計  =', total.sum())

# 廣播：陣列 × 單一數字
discounted = prices * 0.9   # 全部打 9 折
print('\n打折後 =', discounted)

In [ ]:
# 🔥 效能比較：for-loop vs 向量化
import time

big = np.random.rand(1_000_000)

# 方法 1：for-loop（慢）
t0 = time.time()
result1 = [x * 2 + 1 for x in big]
t1 = time.time()
print(f'for-loop   耗時: {(t1-t0)*1000:.1f} ms')

# 方法 2：向量化（快）
t0 = time.time()
result2 = big * 2 + 1
t1 = time.time()
print(f'向量化     耗時: {(t1-t0)*1000:.1f} ms')

print('\n兩者結果相同嗎？', np.allclose(result1, result2))

**口訣**：看到 for-loop 在跑數字運算，99% 可以改成向量化。


---
## 實務 Case — 電商商品庫存分析

**情境**：行銷部要你快速回答三個問題：
1. 全站總庫存價值（unit_price × stock_qty 加總）是多少？
2. 有幾個商品庫存 < 20（快要缺貨）？
3. 如果所有商品都打 8 折，每件商品的新價格是多少？

我們只用 **NumPy**（Pandas 下節再學）。


In [ ]:
import numpy as np

# 直接用 np.genfromtxt 讀 CSV（跳過標題列）
DATA = '../datasets/ecommerce/products.csv'
prices = np.genfromtxt(DATA, delimiter=',', skip_header=1, usecols=3)  # unit_price
stocks = np.genfromtxt(DATA, delimiter=',', skip_header=1, usecols=4)  # stock_qty

print('商品數 =', len(prices))
print('前 5 筆價格:', prices[:5])
print('前 5 筆庫存:', stocks[:5])

In [ ]:
# Q1: 全站總庫存價值
inventory_value = prices * stocks          # 向量化：每個商品的庫存值
total_value = inventory_value.sum()
print(f'Q1 ▸ 全站庫存總值 = NT$ {total_value:,.0f}')

# Q2: 庫存 < 20 的商品數（布林遮罩）
low_stock = stocks < 20
print(f'Q2 ▸ 快缺貨商品數 = {low_stock.sum()} 件')
print(f'     對應價格範圍: {prices[low_stock].min():.0f} ~ {prices[low_stock].max():.0f}')

# Q3: 全部打 8 折（廣播）
discounted = prices * 0.8
print(f'\nQ3 ▸ 前 5 個打折後價格: {discounted[:5]}')

**回顧**：三個商業問題，各用 **1 行 NumPy** 解決。如果換成 for-loop 至少要寫 15 行以上。


---
## 課堂練習（40 min）

以下三題難度遞增，不會就看回上面的範例。

### 🟢 送分題
建立一個 `np.array`，內容是 `[10, 20, 30, 40, 50]`，計算：
1. 所有元素的平均
2. 所有元素乘以 2 之後的結果
3. 篩出大於 25 的元素


In [ ]:
# TODO: 你的程式碼


### 🟡 核心題
使用 `products.csv`，算出：
1. 單價 > 1000 的商品總共有幾個？
2. 庫存最多的前 3 個商品的**索引位置**（提示：`np.argsort`）
3. 假設「單價 < 500 的商品」全部進貨 50 個，這些商品總共要花多少錢？


In [ ]:
# TODO: 你的程式碼


### 🔴 挑戰題
為了備戰雙 11，公司規則：
- 庫存 ≥ 100：打 7 折
- 庫存 20~99：打 9 折
- 庫存 < 20：原價（快缺貨不能再便宜）

請用**向量化**（不要 for-loop）算出每個商品的雙 11 售價。  
提示：`np.where(條件, 值A, 值B)` 可以**巢狀**使用。


In [ ]:
# TODO: 你的程式碼


---
## 小結與速查表

| 想做什麼 | 怎麼寫 |
|---|---|
| 建立陣列 | `np.array([1,2,3])` |
| 看形狀/型別 | `arr.shape`, `arr.dtype` |
| 切片 | `arr[1:4]`, `arr[::2]` |
| 布林篩選 | `arr[arr > 10]` |
| 多條件 | `arr[(arr>10) & (arr<50)]` |
| 加減乘除 | `a + b`, `a * 2`（向量化，無需 loop）|
| 統計 | `.sum() .mean() .max() .min() .std()` |
| 條件賦值 | `np.where(cond, a, b)` |
| 排序索引 | `np.argsort(arr)` |

**下節預告 S2**：今天的 NumPy 是「數字矩陣」，但真實資料有字串、日期、缺值 → **Pandas 登場**，我們會拿一份**故意弄髒的訂單 CSV**，學怎麼把它清乾淨。
